In [ ]:
from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, StringType, StructField, StructType, TimestampType


dbutils.widgets.text("catalog_name", "weather_cat")
CATALOG = dbutils.widgets.get("catalog_name")

spark.sql(f"USE CATALOG {CATALOG}")

# 1. Fetch latest run metadata from Bronze as fallback for interactive or partial (--only) runs
latest_run = spark.sql(f"""
    SELECT run_id,
           MIN(ingestion_ts) AS start_ts,
           COUNT(*) AS attempted,
           SUM(CASE WHEN http_status = 200 THEN 1 ELSE 0 END) AS succeeded
    FROM {CATALOG}.bronze.weather_raw
    GROUP BY run_id
    ORDER BY start_ts DESC
    LIMIT 1
""").first()

if not latest_run:
    raise RuntimeError("No records found in bronze.weather_raw. Run 02_ingest_bronze first!")

fallback_run_id = latest_run["run_id"]
fallback_start_str = latest_run["start_ts"].isoformat()
fallback_attempted = int(latest_run["attempted"])
fallback_succeeded = int(latest_run["succeeded"])

# 2. Retrieve Workflow taskValues with BOTH default and debugValue fallbacks
RUN_ID = dbutils.jobs.taskValues.get(
    taskKey="ingest_bronze",
    key="run_id",
    default=fallback_run_id,
    debugValue=fallback_run_id,
)
START_TIME = datetime.fromisoformat(
    dbutils.jobs.taskValues.get(
        taskKey="ingest_bronze",
        key="start_time",
        default=fallback_start_str,
        debugValue=fallback_start_str,
    )
)
LOC_ATTEMPTED = int(
    dbutils.jobs.taskValues.get(
        taskKey="ingest_bronze",
        key="locations_attempted",
        default=fallback_attempted,
        debugValue=fallback_attempted,
    )
)
LOC_SUCCEEDED = int(
    dbutils.jobs.taskValues.get(
        taskKey="ingest_bronze",
        key="locations_succeeded",
        default=fallback_succeeded,
        debugValue=fallback_succeeded,
    )
)

fallback_silver_count = (
    spark.table(f"{CATALOG}.silver.weather_hourly")
    .filter(F.col("run_id") == RUN_ID)
    .count()
)
SILVER_MERGED = int(
    dbutils.jobs.taskValues.get(
        taskKey="transform_silver",
        key="silver_rows_merged",
        default=fallback_silver_count,
        debugValue=fallback_silver_count,
    )
)

ModuleNotFoundError: No module named 'delta'

In [0]:
# Aggregate Silver into Daily City Insights
gold_updates_df = (
    spark.table(f"{CATALOG}.silver.weather_hourly")
    .withColumn("forecast_date", F.to_date("forecast_ts"))
    .groupBy("forecast_date", "location_id", "city_name")
    .agg(
        F.round(F.min("temperature_2m"), 2).alias("min_temp_c"),
        F.round(F.max("temperature_2m"), 2).alias("max_temp_c"),
        F.round(F.avg("temperature_2m"), 2).alias("avg_temp_c"),
        F.round(F.min("relative_humidity_2m"), 1).alias("min_humidity_pct"),
        F.round(F.max("relative_humidity_2m"), 1).alias("max_humidity_pct"),
        F.round(F.avg("relative_humidity_2m"), 1).alias("avg_humidity_pct"),
        F.round(F.sum("precipitation"), 2).alias("total_precipitation_mm"),
        F.round(F.max("wind_speed_10m"), 2).alias("max_wind_speed_kmh"),
    )
    .withColumn("is_rainy_day", F.col("total_precipitation_mm") >= 1.0)
    .withColumn(
        "weather_alert",
        F.when((F.col("max_temp_c") >= 35.0) & (F.col("avg_humidity_pct") >= 80.0), "EXTREME_HEAT")
        .when(F.col("max_wind_speed_kmh") >= 50.0, "HIGH_WIND")
        .when(F.col("total_precipitation_mm") >= 20.0, "HEAVY_RAIN")
        .otherwise("NORMAL"),
    )
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
# Idempotent MERGE into Gold
gold_target = DeltaTable.forName(spark, f"{CATALOG}.gold.daily_weather_summary")
(
    gold_target.alias("t")
    .merge(
        gold_updates_df.alias("s"),
        "t.forecast_date = s.forecast_date AND t.location_id = s.location_id",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Calculate total quarantined rows for this run and write final Run Audit
quarantined_count = (
    spark.table(f"{CATALOG}.quarantine.weather_quarantine")
    .filter(F.col("run_id") == RUN_ID)
    .count()
)
end_time = datetime.now(timezone.utc)

if LOC_SUCCEEDED == LOC_ATTEMPTED and quarantined_count == 0:
    final_status = "SUCCESS"
elif LOC_SUCCEEDED > 0:
    final_status = "PARTIAL_SUCCESS"
else:
    final_status = "FAILED"

params_snapshot = str(
    {
        r["param_key"]: r["param_value"]
        for r in spark.table(f"{CATALOG}.config.pipeline_params").collect()
    }
)

audit_schema = StructType(
    [
        StructField("run_id", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("parameters", StringType(), True),
        StructField("locations_attempted", IntegerType(), True),
        StructField("locations_succeeded", IntegerType(), True),
        StructField("silver_rows_merged", LongType(), True),
        StructField("quarantined_rows", LongType(), True),
        StructField("error_message", StringType(), True),
    ]
)

run_record = [
    (
        RUN_ID,
        START_TIME,
        end_time,
        final_status,
        params_snapshot,
        LOC_ATTEMPTED,
        LOC_SUCCEEDED,
        SILVER_MERGED,
        quarantined_count,
        None
        if final_status == "SUCCESS"
        else "Check audit.location_audit and quarantine.weather_quarantine",
    )
]

spark.createDataFrame(run_record, audit_schema).write.mode("append").saveAsTable(
    f"{CATALOG}.audit.pipeline_runs"
)

In [0]:
gold_count = spark.table(f"{CATALOG}.gold.daily_weather_summary").count()
dbutils.notebook.exit(
    f"Gold & Audit Complete | Catalog: {CATALOG} | Run ID: {RUN_ID} | Status: {final_status} | "
    f"Silver Merged: {SILVER_MERGED} | Quarantined: {quarantined_count} | Total Gold Rows: {gold_count}"
)
